# 99 · 기능 ↔ 데이터 매핑 보고서

| | |
|---|---|
| **목적** | `reports/*.json` 을 전부 취합해 **기능별 데이터 사용 현황**을 표로 만들고 보고서를 생성 |
| **input** | `data-analysis/reports/*_decisions.json` (6개) + `merge_decisions.json` |
| **output** | `data-analysis/reports/preprocessing_report.md` |

> 📌 요청서의 `{project}_core_features.md` 는 **이 저장소에 없다.**
> 대응 문서는 **`docs/00_MVP_SCOPE.md` 의 기능 1~13 표**이고, 아래 `FEATURES` 는 거기서 옮긴 것이다.

In [ ]:
import sys
sys.path.insert(0, "..")
import json
from datetime import date

import pandas as pd
from common import REPORTS, REPO_ROOT

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)

## 1. 결정 로그 취합

In [ ]:
decisions = {}
for p in sorted(REPORTS.glob("*_decisions.json")):
    if p.stem == "merge_decisions":
        continue
    decisions[p.stem.replace("_decisions", "")] = json.loads(p.read_text(encoding="utf-8"))
merge = json.loads((REPORTS / "merge_decisions.json").read_text(encoding="utf-8"))
print(f"데이터셋 결정 로그 {len(decisions)}건 + merge 1건")
for k in decisions:
    print("  ", k)

## 2. 기능 정의 — `docs/00_MVP_SCOPE.md` 기준

기능 1~6 = 코어, 7~13 = 확장(D67). **어느 데이터셋을 쓰는지**를 이 노트북이 매핑한다.

In [ ]:
FEATURES = {
    1: "에러코드 진단 — 룩업 × RAG 이원화",
    2: "부품 특정 → 재고 · 견적",
    3: "발주서 초안 + 승인 워크플로우",
    4: "반복 고장 감지",
    5: "안전 가드레일",
    6: "실행 trace 시각화",
    7: "근거 3계층 — 법령 · 해석 · 서명",
    8: "처분 법정 조건 검사",
    9: "처분서 초안 + 사람 서명",
    10: "보전지표 — MTBF · 예방보전 비율",
    11: "수리 / 교체 / 매각 3지 판단",
    12: "중고 취득 권리관계 검증",
    13: "수리 증빙 서명 (Sprint 8)",
}

# 데이터셋 → 사용 기능. 근거는 data_list.md · maintq-data-map.html
DATASET_FEATURES = {
    "error_codes": [1, 2, 4],
    "manual_chunks": [1, 5],
    "joonggomall_assets": [11, 12],
    "kseis_machinery": [2, 11],
    "kseis_electrical": [2, 11],
    "iros_collateral_stats": [12],
    "law_articles": [7, 8, 9],
    "dacon_fan_sound": [],          # 미채택
}
SOURCES = {
    "error_codes": ("LS 매뉴얼 표 추출 (iG5A·S100)", "sol.ls-electric.com"),
    "manual_chunks": ("LS 매뉴얼 청킹 (PDF 3종)", "sol.ls-electric.com"),
    "joonggomall_assets": ("중진공 자산거래중개장터", "data.go.kr/15112833"),
    "kseis_machinery": ("조달청 시설공통자재(기계설비)", "kseis.co.kr"),
    "kseis_electrical": ("조달청 시설공통자재(전기,정보통신)", "kseis.co.kr"),
    "iros_collateral_stats": ("법원 등기정보광장", "data.iros.go.kr"),
    "law_articles": ("국가법령정보 (법제처)", "open.law.go.kr"),
    "dacon_fan_sound": ("DACON 236036 팬 소음", "dacon.io/236036"),
}
print(f"기능 {len(FEATURES)}종 · 데이터셋 {len(DATASET_FEATURES)}종")

## 3. 데이터셋 요약표

In [ ]:
gen_by_ds = {}
for g in merge["generated_columns"]:
    for ds in g["dataset"].split(" / "):
        gen_by_ds.setdefault(ds.strip(), []).append(g["column"])

rows = []
for ds, d in decisions.items():
    이름, 출처 = SOURCES[ds]
    기능 = DATASET_FEATURES[ds]
    파생 = len(d.get("generated_columns", [])) + len(gen_by_ds.get(ds, []))
    rows.append({
        "데이터셋": ds,
        "출처": 이름,
        "원본 행": f"{d.get('row_count_raw', 0):,}",
        "원본 컬럼": d.get("col_count_raw", d.get("column_total", 0)),
        "사용 컬럼": d["column_used"],
        "생성 컬럼": 파생,
        "사용 기능": ", ".join(map(str, 기능)) if 기능 else "— (미채택)",
    })
summary = pd.DataFrame(rows)
summary

## 4. 기능 → 데이터셋 역매핑

In [ ]:
rev = []
for f, name in FEATURES.items():
    ds = [d for d, fs in DATASET_FEATURES.items() if f in fs]
    rev.append({
        "기능": f,
        "이름": name,
        "이 분석이 다루는 데이터셋": ", ".join(ds) if ds else "— (이 분석 범위 밖)",
    })
pd.DataFrame(rev)

> 📌 **아직 비어 있는 기능 3·6·10·13 은 원천이 전부 `seed.py` 목업 DB** 다
> (`po_drafts`·`users`·`traces`·`error_history`·`repair_records`).
> **외부에서 받은 데이터가 아니라 우리가 만든 것**이라 EDA·전처리 대상이 아니다 —
> 규모·근거는 `docs/status/maintq-data-map.html` 의 "생성한 데이터" 절이 관리한다.

## 5. 이 분석이 실제로 잡아낸 것

In [ ]:
findings = []
for ds, d in decisions.items():
    for c in d.get("caveats", []):
        findings.append({"출처": ds, "종류": "caveat", "내용": c})
    if d.get("discrepancy_note"):
        findings.append({"출처": ds, "종류": "문서 불일치", "내용": d["discrepancy_note"]})
for f in merge["findings"]:
    findings.append({"출처": "90_merge", "종류": "finding", "내용": f})
fdf = pd.DataFrame(findings)
print(f"총 {len(fdf)}건")
fdf

## 6. `preprocessing_report.md` 생성

In [ ]:
def md_table(df):
    head = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "|".join("---" for _ in df.columns) + "|"
    body = ["| " + " | ".join(str(v) for v in r) + " |" for r in df.to_numpy()]
    return "\n".join([head, sep, *body])


L = []
L.append("# MaintQ 전처리 보고서\n")
L.append(f"> 자동 생성 — `data-analysis/notebooks/99_feature_mapping_report.ipynb`\n"
         f"> 생성일 **{date.today().isoformat()}**\n")
L.append("이 보고서는 **데이터셋 8종**(외부 원천 6 + 매뉴얼 파생 2)의 EDA·전처리 결과다. "
         "기능 정의는 `docs/00_MVP_SCOPE.md` 의 기능 1~13 표를 따른다.\n")

L.append("\n## 1. 데이터셋 요약\n")
L.append(md_table(summary))

L.append("\n\n## 2. 기능 → 데이터셋\n")
L.append(md_table(pd.DataFrame(rev)))
L.append("\n\n> 비어 있는 기능 **3·6·10·13** 의 원천은 전부 `seed.py` 목업 DB 다 "
         "(`po_drafts`·`users`·`traces`·`error_history`·`repair_records`). "
         "**외부에서 받은 데이터가 아니라 우리가 만든 것**이라 EDA·전처리 대상이 아니다 — "
         "규모·근거는 `docs/status/maintq-data-map.html` 의 \"생성한 데이터\" 절이 관리한다.\n")

L.append("\n## 3. 병합\n")
for jn in merge["joins"]:
    L.append(f"\n### {jn['name']}\n")
    L.append(f"- **방식**: {jn['method']}")
    L.append(f"- **키**: `{jn['join_key']}`" if jn["join_key"] else "- **키**: 없음")
    L.append(f"- **이유**: {jn['reason']}")
    before = jn["rows_before"]
    before_s = (", ".join(f"{k} {v:,}" for k, v in before.items())
                if isinstance(before, dict) else f"{before:,}")
    L.append(f"- **행 수**: {before_s} → **{jn['rows_after']:,}행**")

L.append("\n### 합치지 않은 데이터셋\n")
for k, v in merge["not_joined"].items():
    L.append(f"- `{k}` — {v}")

L.append("\n\n## 4. 생성한 파생 컬럼\n")
gdf = pd.DataFrame(merge["generated_columns"])[["dataset", "column", "logic", "source_columns"]]
gdf["source_columns"] = gdf["source_columns"].map(lambda x: ", ".join(f"`{c}`" for c in x))
gdf.columns = ["데이터셋", "생성 컬럼", "로직", "원본 컬럼"]
L.append(md_table(gdf))

L.append("\n\n## 5. 발견 사항 · 주의\n")
L.append(md_table(fdf))

L.append("\n\n## 6. 컬럼 결정 상세\n")
for ds, d in decisions.items():
    L.append(f"\n### {ds} — {d['column_used']}/{d['column_total']} 사용\n")
    cd = pd.DataFrame(d["column_decisions"])
    cd["used"] = cd["used"].map({True: "✅", False: "—"})
    cd.columns = ["컬럼", "사용", "이유"]
    L.append(md_table(cd))

report = "\n".join(L) + "\n"
path = REPORTS / "preprocessing_report.md"
path.write_text(report, encoding="utf-8")
print(f"✅ {path.relative_to(REPO_ROOT)}  ({len(report):,}자 · {report.count(chr(10)):,}줄)")

### 미리보기

In [ ]:
print(report[:2200])